**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**03 | Full-Cohort Numeric and Temporal Validation**

![Full-cohort numerical and original time-coordinate workflow](../../images/03_consolidated_workflow.svg)

**Research objective:** Verify source-linked numerical IMU and FE signals across the complete original Essex dataset and match original FE time coordinates to observed IMU indices. The analytical unit remains an original movement trial.

**Dataset and upstream evidence:** Original Essex [Figshare 30225739 v13](https://doi.org/10.6084/m9.figshare.30225739.v13), verified in [Notebook 01](01_original_dataset_audit_and_trial_registry.ipynb); decoder and original schema in [Notebook 02](02_matlab_object_decoding_and_numeric_recovery.ipynb). The committed 01/02 Quality Gates, CSV registry and SHA256 are checked afresh.

**Why this integration matters:** Historical Stage 06 verified 1,897 original source records with 42 finite IMU channels, including 1,729 finite numerical e1/e3 FE records, but its fixed `IMU[::10]` time-grid assumption differed for all 1,729 records. Historical Stage 07 matched **the original source `timestep` coordinates**, yielding 1,729 exact mappings and classifying all 168 original FE-empty records. These results answer different questions and must not be conflated.

**Illustrative mapped positions:** The original time vector for an observed trial maps the eleven FE values to zero-based IMU positions `[0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]`, not a presumed evenly-spaced `[0, 10, ..., 100]` grid.

**Scope and limits:** Numeric channel validation, source identities, native time coordinates and FE missingness only. No interpolation, synthetic targets, full numeric data export, physical-unit validation, participant-disjoint regression evaluation or model training. The new notebook is **not yet executed**; the observations above come from previously executed, preserved Stages 06 and 07.

**Outputs:** All new artefacts use unique `03_*` names directly under `biomechanical_load_estimation/output/`. The conventional stage-03 Colab download is included once in the final Quality Gate; no separate packaging notebook cell.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.split("/")[-1].removesuffix(".git")

%cd /content

if not repo_path.exists():
    !git clone -q {repo_url}

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

Use the published source loader, original numeric audit, native coordinate mapping and standard plotting/output helpers.


In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("biomechanical_load_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import json
from importlib.metadata import version
from pathlib import Path

# -------------------- 3. Numerical and Tabular Computing ---------------------
import numpy as np
import pandas as pd

# ---------------------- 4. Figures and Notebook Display ----------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5, label_weight="normal",
    tick_size=11, legend_size=10, figure_dpi=100,
    figure_size=(9, 4))

# ------------- 5. Source, Timebase and Temporal Mapping Modules --------------
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID, ARTICLE_VERSION, fetch_version_manifest,
    find_published_file, ensure_source_file,
    inspect_matlab_archive, extract_matlab_files,
)
from essex_numeric_recovery import check_record_registry
from essex_timebase_audit import (
    EXPECTED_CHANNELS, COMPONENTS, SENSORS,
    audit_original_source, summarise_alignment,
)
from essex_temporal_mapping import (
    audit_temporal_mapping, summarise_temporal_evidence,
)

# ------------------ 6. Dataset and Flat Output Directories -------------------
study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = study_dir / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
assert output_dir.name == "output"
print("Original Essex source:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("Stage 03 outputs:", output_dir)


**02. Pinned Python MATLAB Table Reader**

Keep the tested `mat-io==1.0.1` decoder version and report it before inspecting original MATLAB table values.


In [ ]:
# =================== 02. Pinned Python MATLAB Table Reader ===================

%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"

from matio import load_from_mat

assert version("mat-io") == "1.0.1"
print("Python MATLAB reader:", version("mat-io"))


**03. Dataset Integrity and Original Trial Registry**

Check committed 01/02 Quality Gates, original record identities and the official source archive SHA256, avoiding dependencies on retired 02/05 outputs.


In [ ]:
# ============= 03. Dataset Integrity and Original Trial Registry =============

# ---------- 1. Load the Committed Upstream Scientific Quality Gates ----------
source_gate_path = output_dir / "01_consolidated_source_and_trial_quality_gate.json"
decoder_gate_path = output_dir / "02_consolidated_decoding_quality_gate.json"
registry_path = output_dir / "01_original_record_registry.csv"
if not all(p.is_file() for p in (
    source_gate_path, decoder_gate_path, registry_path
)):
    raise FileNotFoundError("Run and commit Notebook 01/02 outputs first.")
source_gate = json.loads(source_gate_path.read_text(encoding="utf-8"))
decoder_gate = json.loads(decoder_gate_path.read_text(encoding="utf-8"))
assert source_gate["source_article_id"] == ARTICLE_ID == 30225739
assert source_gate["version"] == ARTICLE_VERSION == 13
assert source_gate["source_and_trial_quality_gate"] == "PASSED"
assert decoder_gate["decoding_consistency_gate"] == "PASSED"
assert decoder_gate["source_sha256"] == source_gate["archive_sha256"]

# ----------- 2. Retrieve and Verify the Original Published Archive -----------
_, source_files = fetch_version_manifest()
archive_item = find_published_file(source_files, "MATLAB.zip")
source_zip, source_hash = ensure_source_file(archive_item, raw_dir)
assert source_hash["sha256"] == source_gate["archive_sha256"]
archive_inventory = inspect_matlab_archive(
    source_zip, expected_mat_files=40
)
mat_files = extract_matlab_files(
    source_zip, mat_dir, archive_inventory
)
assert len(mat_files) == 40

# ---------------- 3. Verify the Full Original Trial Registry -----------------
registry = pd.read_csv(registry_path, dtype={
    "subject_id": "string", "source_file": "string",
    "local_record_key": "string", "task": "string",
    "updated_trial_id": "string", "fe_representation": "string",
})
registry_status = check_record_registry(registry, source_gate)
assert registry_status["records"] == 1897
assert registry_status["participants"] == 40
assert registry["fe_representation"].eq("empty").sum() == 168
assert {p.stem.lower() for p in mat_files} == set(registry["subject_id"])
print("Validated original records:", len(registry))
print("Validated original participant files:", len(mat_files))
print("Original archive SHA256:", source_hash["sha256"])


**04. Original IMU and FE Column Contract**

Verify 42 original sensor channels, the original FE `e1/e3` labels and finite pilot schema data without inferring measurement units.


In [ ]:
# ================== 04. Original IMU and FE Column Contract ==================

schema = pd.read_csv(
    output_dir / "02_python_column_schema.csv"
)
imu_schema = schema.loc[schema["modality"] == "IMU"]
fe_schema = schema.loc[schema["modality"] == "FE"]
imu_names = set(imu_schema["original_column_name"])
fe_names = set(fe_schema["original_column_name"])
assert imu_names == set(EXPECTED_CHANNELS) | {"timestep"}
assert fe_names == {"timestep", "e1", "e3"}
assert len(EXPECTED_CHANNELS) == 42
assert len(COMPONENTS) == 6 and len(SENSORS) == 7
assert imu_schema.loc[
    imu_schema["original_column_name"] != "timestep",
    "is_real_finite_numeric"
].all()
assert fe_schema.loc[
    fe_schema["original_column_name"].isin(["e1", "e3"]),
    "is_real_finite_numeric"
].all()

channel_contract = pd.DataFrame([
    {"original_column_name": label,
     "signal_component": label.rsplit("_", 1)[0],
     "sensor_code": label.rsplit("_", 1)[1]}
    for label in EXPECTED_CHANNELS
])
channel_contract.to_csv(
    output_dir / "03_imu_channel_contract.csv", index=False
)
display(channel_contract.head(14))
print("Original numeric IMU channel names:", len(channel_contract))


**05. Complete Numerical Record Audit**

Use the source-linked `audit_original_source` function to inspect 1,897 records, IMU channels, FE values, reader failures and the original time axes.


In [ ]:
# ==================== 05. Complete Numerical Record Audit ====================

# ======== 06. Audit Numerical Channels and Timebases for All Records =========

record_audit, reader_errors, example = audit_original_source(
    load_from_mat, mat_dir, registry
)
assert len(record_audit) == 1897
assert record_audit["local_record_key"].is_unique

record_audit.to_csv(
    output_dir / "03_original_numeric_record_audit.csv", index=False
)
reader_errors.to_csv(
    output_dir / "03_reader_errors.csv", index=False
)
print("Full original source records audited:", len(record_audit))
print("Files with reader errors:", len(reader_errors))
display(record_audit[[
    "subject_id", "task", "imu_status", "fe_status",
    "imu_timestep_status", "fe_timestep_status",
    "time_alignment",
]].head(20))


**06. Numeric Coverage across Participants and Activities**

Summarise validated numerical IMU/FE coverage and original FE-empty status for each participant and movement task.


In [ ]:
# ========== 06. Numeric Coverage across Participants and Activities ==========

def coverage(frame, by):
    return frame.groupby(by).agg(
        source_records=("local_record_key", "size"),
        valid_identities=("identity_valid", "sum"),
        numeric_imu=("imu_status", lambda s: s.eq("VALID_CHANNELS").sum()),
        numeric_fe=("fe_status", lambda s: s.eq("VALID_STRAIN_COLUMNS").sum()),
        source_fe_empty=("fe_status", lambda s: s.eq("EXPECTED_EMPTY").sum()),
        time_grid_matches=("time_alignment",
                           lambda s: s.eq("NUMERIC_TIME_GRID_MATCH").sum()),
    ).reset_index()

participant_coverage = coverage(record_audit, "subject_id")
activity_coverage = coverage(record_audit, "task")
participant_coverage.to_csv(
    output_dir / "03_participant_numeric_coverage.csv", index=False
)
activity_coverage.to_csv(
    output_dir / "03_activity_numeric_coverage.csv", index=False
)
display(activity_coverage)
print("Participant rows:", len(participant_coverage))


**07. Fixed-Index Timebase Comparison**

Preserve and diagnose the limited `IMU[::10]` comparison and observed source time-axis metadata, rather than assuming it proves that valid matching is impossible.


In [ ]:
# ==================== 07. Fixed-Index Timebase Comparison ====================

# ====== 08. Summarise Original Timebase Classes and Grid Correspondence ======

alignment_counts = (
    record_audit.groupby(
        ["fe_status", "time_alignment"], dropna=False
    ).size().rename("records").reset_index()
)
alignment_counts.to_csv(
    output_dir / "03_time_alignment_status.csv", index=False
)
time_cols = [
    "local_record_key", "subject_id", "task",
    "imu_timestep_status", "fe_timestep_status",
    "fe_time_entry_class", "imu_t0", "imu_tlast", "imu_tstep",
    "fe_t0", "fe_tlast", "fe_tstep",
    "time_alignment", "time_error",
]
for col in time_cols:
    if col not in record_audit:
        record_audit[col] = np.nan
time_examples = record_audit.loc[:, time_cols].copy()
time_examples.to_csv(
    output_dir / "03_timebase_metadata.csv", index=False
)
display(alignment_counts)
print("FE and IMU time vectors must agree before paired export.")


**08. Native FE-to-IMU Time-Coordinate Mapping**

Match the original source FE `timestep` values directly against 101-point IMU coordinates and record unique zero-based sample indices for every source trial.


In [ ]:
# =============== 08. Native FE-to-IMU Time-Coordinate Mapping ================

# ======== 05. Audit All Original Time Coordinates and FE Empty Fields ========

mapping, fe_empty, temporal_reader_errors, source_example = (
    audit_temporal_mapping(load_from_mat, mat_dir, registry)
)
assert len(mapping) == 1897
assert mapping["local_record_key"].is_unique
assert len(fe_empty) == 168

mapping.to_csv(
    output_dir / "03_original_time_coordinate_mapping.csv", index=False
)
fe_empty.to_csv(
    output_dir / "03_source_fe_empty_representations.csv", index=False
)
temporal_reader_errors.to_csv(
    output_dir / "03_source_reader_errors.csv", index=False
)
print("All original movement records:", len(mapping))
print("Original FE-empty representations inspected:", len(fe_empty))
print("MAT reader errors:", len(temporal_reader_errors))
display(mapping[[
    "source_file", "task", "fe_channel_status",
    "mapping_status", "imu_position_indices",
]].head(20))


**09. Source FE Missingness and Reader Empty Classes**

Reconcile all 168 source-declared FE-empty records, including reader-specific canonical empty and empty ndarray representations, without converting them into strain targets.


In [ ]:
# ============ 09. Source FE Missingness and Reader Empty Classes =============

missingness = (
    fe_empty.groupby([
        "python_class", "python_shape",
        "empty_representation_status", "column_names",
        "fe_numeric_columns_finite",
    ], dropna=False)
    .size().rename("source_records").reset_index()
)
missingness.to_csv(
    output_dir / "03_fe_missingness_types.csv", index=False
)
display(missingness)
print("Total source-declared FE empties:",
      missingness["source_records"].sum())


**10. Original Mapping Signatures and Cohort Coverage**

Aggregate exact coordinate matches, source-empty FE cases and mapping indices; report outcomes separately for participants and activity categories.


In [ ]:
# ============ 10. Original Mapping Signatures and Cohort Coverage ============

mapping_signatures = (
    mapping.groupby([
        "mapping_status", "consecutive_index_steps",
    ], dropna=False).size().rename("source_records").reset_index()
)
mapping_signatures.to_csv(
    output_dir / "03_time_mapping_signatures.csv", index=False
)
display(mapping_signatures.head(30))
print("Original mapping status distribution:")
print(mapping["mapping_status"].value_counts())

# ------------ 2. Coordinate Coverage by Participant and Activity -------------
def group_coverage(source, by):
    return source.groupby(by).agg(
        original_records=("local_record_key", "size"),
        source_empty_fe=("originally_empty_fe", "sum"),
        coordinate_mappings=(
            "mapping_status",
            lambda s: s.isin([
                "EXACT_TIME_COORDINATE_MAPPING",
                "TOLERANT_TIME_COORDINATE_MAPPING",
            ]).sum(),
        ),
        unresolved_time=(
            "mapping_status",
            lambda s: s.eq("TIME_VALUES_NOT_AVAILABLE").sum(),
        ),
    ).reset_index()

participant_report = group_coverage(mapping, "subject_id")
activity_report = group_coverage(mapping, "task")
participant_report.to_csv(
    output_dir / "03_participant_coordinate_coverage.csv", index=False
)
activity_report.to_csv(
    output_dir / "03_activity_coordinate_coverage.csv", index=False
)
display(activity_report)
print("Participants reported:", len(participant_report))


**11. Full Numeric Coverage and Source Signal Figures**

Show per-activity finite channels, fixed-grid classifications and authentic numerical IMU and e1/e3 traces. Do not display predicted clinical strain.


In [ ]:
# ============ 11. Full Numeric Coverage and Source Signal Figures ============

# ======== 09. Source Coverage, Time Grid and Authentic Signal Figures ========

fig, axes = plt.subplots(
    1, 3, figsize=(18, 5.1),
    gridspec_kw={"width_ratios": [1.7, 1.25, 1.0]},
    constrained_layout=True,
)
tasks = activity_coverage["task"]
positions = np.arange(len(tasks))
axes[0].bar(
    positions - 0.16, activity_coverage["numeric_imu"],
    width=0.31, label="42 IMU channels"
)
axes[0].bar(
    positions + 0.16, activity_coverage["numeric_fe"],
    width=0.31, label="FE e1 and e3"
)
axes[0].set_xticks(positions)
axes[0].set_xticklabels(
    [name.replace("Hopping", " Hopping").replace("Jumps", " Jumps")
     for name in tasks], rotation=25, ha="right"
)
axes[0].set_title("Numeric source coverage by activity")
axes[0].set_ylabel("Original records")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

statuses = (
    record_audit.groupby("time_alignment")
    .size().sort_values(ascending=True)
)
yy = np.arange(len(statuses))
axes[1].barh(yy, statuses.values, height=0.34)
axes[1].set_yticks(yy)
axes[1].set_yticklabels(
    [s.replace("_", " ").title() for s in statuses.index]
)
axes[1].set_title("IMU–FE time-grid classifications")
axes[1].set_xlabel("Original records")
axes[1].grid(axis="x", alpha=0.2)

if example is not None and example["imu_time"] is not None:
    axes[2].plot(example["imu_time"], np.zeros(101), ".", label="IMU grid")
    if example["fe_time"] is not None:
        axes[2].plot(
            example["fe_time"], np.ones(11), "o", markersize=4,
            label="FE grid"
        )
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(["IMU", "FE"])
    axes[2].set_title("Original source time coordinates")
    axes[2].set_xlabel("Original time coordinate")
    axes[2].legend(frameon=False)
else:
    axes[2].text(
        0.5, 0.5, "No numeric source time axis available",
        ha="center", va="center", transform=axes[2].transAxes
    )
    axes[2].set_title("Original time-axis availability")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
save_figure(fig, output_dir / "03_numeric_alignment_diagnostics.png")
plt.show()

fig2, axs = plt.subplots(1, 2, figsize=(13.5, 4.5),
                         constrained_layout=True)
if example is not None:
    axs[0].plot(
        np.arange(len(example["imu_signal"])),
        example["imu_signal"], linewidth=1.4
    )
    axs[0].set_title("Original IMU: " + example["imu_name"])
    axs[0].set_ylabel("Source numerical value")
    axs[0].set_xlabel("IMU sample index")
    axs[1].plot(np.arange(len(example["e1"])), example["e1"],
                label="e1", linewidth=1.4)
    axs[1].plot(np.arange(len(example["e3"])), example["e3"],
                label="e3", linewidth=1.4)
    axs[1].set_title("Original FE strain columns")
    axs[1].set_xlabel("FE sample index")
    axs[1].set_ylabel("Source strain value")
    axs[1].legend(frameon=False)
else:
    for axis in axs:
        axis.text(0.5, 0.5, "No validated numeric example",
                  ha="center", va="center", transform=axis.transAxes)
        axis.set_xticks([])
        axis.set_yticks([])
save_figure(fig2, output_dir / "03_original_numeric_examples.png")
plt.show()


**12. Original Time-Coordinate and FE Coverage Figures**

Use observed source coordinates and mapped index positions to visualise FE/IMU mapping and separately chart original FE-empty representation types.


In [ ]:
# =========== 12. Original Time-Coordinate and FE Coverage Figures ============

fig, axes = plt.subplots(
    1, 3, figsize=(18, 5.1),
    gridspec_kw={"width_ratios": [1.6, 1.1, 1.1]},
    constrained_layout=True,
)
pos = np.arange(len(activity_report))
axes[0].bar(
    pos - 0.17, activity_report["coordinate_mappings"],
    width=0.33, label="Mapped FE time vectors",
)
axes[0].bar(
    pos + 0.17, activity_report["source_empty_fe"],
    width=0.33, label="Source-empty FE",
)
axes[0].set_xticks(pos)
axes[0].set_xticklabels(
    activity_report["task"].str.replace(
        "Hopping", " Hopping", regex=False
    ), rotation=26, ha="right",
)
axes[0].set_title("Original FE coordinate coverage")
axes[0].set_ylabel("Movement records")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

state_counts = mapping["mapping_status"].value_counts().sort_values()
axes[1].barh(
    np.arange(len(state_counts)), state_counts.values, height=0.32
)
axes[1].set_yticks(np.arange(len(state_counts)))
axes[1].set_yticklabels(
    [s.replace("_", " ").title() for s in state_counts.index]
)
axes[1].set_title("Original mapping classifications")
axes[1].set_xlabel("Movement records")
axes[1].grid(axis="x", alpha=0.2)

class_counts = (
    fe_empty["empty_representation_status"].value_counts()
)
axes[2].barh(
    np.arange(len(class_counts)), class_counts.values, height=0.32
)
axes[2].set_yticks(np.arange(len(class_counts)))
axes[2].set_yticklabels(
    [s.replace("_", " ").title() for s in class_counts.index]
)
axes[2].set_title("Source-empty FE representations")
axes[2].set_xlabel("Original source-empty records")
axes[2].grid(axis="x", alpha=0.2)
save_figure(fig, output_dir / "03_temporal_mapping_diagnostics.png")
plt.show()

fig2, axs = plt.subplots(1, 2, figsize=(13, 4.5),
                         constrained_layout=True)
if source_example is not None:
    actual_imu = source_example["imu_time"]
    actual_fe = source_example["fe_time"]
    mapped = np.asarray(source_example["imu_indices"])
    axs[0].plot(
        np.arange(len(actual_imu)), actual_imu,
        linewidth=1.2, label="Original IMU time"
    )
    axs[0].scatter(
        mapped, actual_fe, s=24, label="Original FE time", zorder=3
    )
    axs[0].set_title("Native FE coordinates on original IMU positions")
    axs[0].set_xlabel("Original IMU sample index (zero-based)")
    axs[0].set_ylabel("Original time coordinate")
    axs[0].legend(frameon=False)
    axs[1].plot(
        np.arange(len(actual_fe)), mapped, "o-",
        linewidth=1.2
    )
    axs[1].set_title("Source FE-to-IMU sample mapping")
    axs[1].set_xlabel("Original FE sample index")
    axs[1].set_ylabel("Matched IMU sample index")
else:
    for axis in axs:
        axis.text(0.5, 0.5, "No unique original time mapping",
                  ha="center", va="center",
                  transform=axis.transAxes)
        axis.set_xticks([])
        axis.set_yticks([])
save_figure(fig2, output_dir / "03_original_coordinate_example.png")
plt.show()


**13. Integrated Numerical and Temporal Quality Gate**

Combine full-source numerical and native coordinate findings, preserve the 168 original FE-empty records and validate all 19 outputs. A PASS refers to source integrity, not physical units or a prediction-ready dataset.


In [ ]:
# ============ 13. Integrated Numerical and Temporal Quality Gate =============

# -------- 1. Independently Summarise Both Original Scientific Audits ---------
numeric_summary = summarise_alignment(record_audit, reader_errors)
mapping_summary = summarise_temporal_evidence(
    mapping, fe_empty, temporal_reader_errors
)
display(pd.DataFrame([numeric_summary]))
display(pd.DataFrame([mapping_summary]))

# ------ 2. Enforce Original Identity, Numeric and Mapping Quality Gates ------
assert len(registry) == len(record_audit) == len(mapping) == 1897
assert numeric_summary["participants"] == 40
assert mapping_summary["participants"] == 40
assert record_audit["local_record_key"].is_unique
assert mapping["local_record_key"].is_unique
assert set(record_audit["local_record_key"]) == set(registry["local_record_key"])
assert set(mapping["local_record_key"]) == set(registry["local_record_key"])
assert len(reader_errors) == len(temporal_reader_errors) == 0
assert numeric_summary["source_identical"]
assert mapping_summary["source_identity_gate"]
assert numeric_summary["imu_42_channel_records"] == 1897
assert numeric_summary["fe_e1_e3_records"] == 1729
assert mapping_summary["registry_declared_fe_empty"] == 168
assert len(fe_empty) == 168
assert mapping_summary["reader_verified_empty_representations"] == 168
assert mapping_summary["unmatched_nonempty_fe_records"] == 0
assert mapping_summary["exact_or_tolerant_coordinate_mappings"] == 1729
assert mapping_summary["consistent_unique_source_mapping"]
assert mapping_summary["source_fe_empty_resolution_complete"]

# ------- 3. Reconcile Fixed-Grid and Actual Source-Coordinate Results --------
fixed_matches = int(
    record_audit["time_alignment"].eq("NUMERIC_TIME_GRID_MATCH").sum()
)
fixed_differs = int(
    record_audit["time_alignment"].eq("NUMERIC_TIME_GRID_DIFFERS").sum()
)
direct_matches = int(
    mapping["mapping_status"].isin([
        "EXACT_TIME_COORDINATE_MAPPING",
        "TOLERANT_TIME_COORDINATE_MAPPING",
    ]).sum()
)
assert fixed_matches + fixed_differs == 1729
assert direct_matches == 1729
print("Fixed-index matches:", fixed_matches)
print("Fixed-index mismatches:", fixed_differs)
print("Unique native coordinate mappings:", direct_matches)

# ------ 4. Record Scientific Quality and Explicit Research Limitations -------
quality = {
    "source_article_id": ARTICLE_ID,
    "source_version": ARTICLE_VERSION,
    "original_archive_sha256": source_hash["sha256"],
    "reader": "mat-io",
    "reader_version": version("mat-io"),
    "numeric_source_audit": numeric_summary,
    "source_coordinate_and_missingness_audit": mapping_summary,
    "fixed_index_time_grid_matches": fixed_matches,
    "fixed_index_time_grid_differs": fixed_differs,
    "direct_source_coordinate_mappings": direct_matches,
    "full_numeric_temporal_quality_gate": "PASSED",
    "physical_units_independently_verified": False,
    "full_numeric_regression_dataset_exported": False,
    "model_training_performed": False,
    "supervised_regression_ready": False,
}
(output_dir / "03_consolidated_numeric_temporal_quality_gate.json").write_text(
    json.dumps(quality, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8"
)
print("CONSOLIDATED 03 | SOURCE NUMERIC/TEMPORAL GATE: PASSED")
print("Original physical units and model readiness: NOT VERIFIED")

# ---------- 5. Validate Outputs and Package the Standard Colab ZIP -----------
expected = [
    "03_imu_channel_contract.csv",
    "03_original_numeric_record_audit.csv",
    "03_reader_errors.csv",
    "03_participant_numeric_coverage.csv",
    "03_activity_numeric_coverage.csv",
    "03_time_alignment_status.csv",
    "03_timebase_metadata.csv",
    "03_numeric_alignment_diagnostics.png",
    "03_original_numeric_examples.png",
    "03_original_time_coordinate_mapping.csv",
    "03_source_fe_empty_representations.csv",
    "03_source_reader_errors.csv",
    "03_fe_missingness_types.csv",
    "03_time_mapping_signatures.csv",
    "03_participant_coordinate_coverage.csv",
    "03_activity_coordinate_coverage.csv",
    "03_temporal_mapping_diagnostics.png",
    "03_original_coordinate_example.png",
    "03_consolidated_numeric_temporal_quality_gate.json",
]
missing = [name for name in expected if not (output_dir / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing stage-03 research evidence: " + ", ".join(missing)
    )
print("Validated stage-03 files:", len(expected))
download_outputs_from_colab(
    folder="biomechanical_load_estimation/output", stage="03"
)


**14. Scientific Interpretation and Next Research Boundary**

This notebook unifies two previously executed and complementary investigations. The original Stage 06 found 1,897 finite 42-channel IMU records and 1,729 finite e1/e3 records but 1,729 failures under the fixed `IMU[::10]` comparison; its strict reader representation classified only 24 of 168 source-empty FE fields as expected. Historical Stage 07 examined all 168 original empty fields (24 `MatlabCanonicalEmpty` plus 144 empty ndarray representations) and found 1,729 exact direct source-coordinate mappings.

The sample source-derived FE index signature is `[0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]`. The earlier fixed-grid mismatch is therefore a **specific sampling-index assumption failure**, not a failure of the original FE–IMU time relationship.

**Limitations:** No inference of physical strain units, imputation for source-missing FE, full raw-data conversion, clinical deployment, subject-independent regression or predictive model training. Complete coordinate matching does not alone establish regression readiness.

**Next independent research topic:** Governed dataset construction and participant-level evaluation design once target semantics and unit provenance have been assessed.

**Approval status:** Unexecuted consolidated Notebook 03. The executed original Stages 06 and 07 remain intact until the new evidence is generated and independently checked.
